# 4단계 모델링 및 분석

도시별 feature CSV와 RoBERTa PCA CSV를 결합해 불만족 리뷰를 예측합니다. 모델 성능표와 주요 시각화는 `output/figures/`, `output/tables/`에 저장합니다.


In [ ]:
# 0. 나눔고딕 폰트와 필요한 패키지 설치
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

!pip install catboost shap

## 0. 환경 설정

필요 라이브러리를 불러오고 프로젝트 루트 및 `output/` 저장 경로를 설정합니다.


In [ ]:
# 0. 필요한 라이브러리 import 및 폰트 설정하기
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import seaborn as sns
import shap
from pathlib import Path
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from sklearn.metrics import roc_curve, auc

# 폰트 설정
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if Path(font_path).exists():
    fm.fontManager.addfont(font_path)
    plt.rcParams['font.family'] = 'NanumGothic'
else:
    print(f'나눔 폰트를 찾지 못했습니다: {font_path}. 기본 글꼴을 사용합니다.')
plt.rcParams['axes.unicode_minus'] = False # 마이너스 기호 깨짐 방지
shap.initjs() # SHAP 자바스크립트 시각화 활성화

# 프로젝트 경로와 산출물 저장 경로
PROJECT_ROOT = Path('/content/drive/MyDrive/ml_project/Team-6')
if not PROJECT_ROOT.exists():
    current_dir = Path.cwd().resolve()
    for candidate in [current_dir, *current_dir.parents]:
        if (candidate / 'data').exists() and (candidate / 'notebooks').exists():
            PROJECT_ROOT = candidate
            break
    else:
        raise FileNotFoundError('프로젝트 루트를 찾을 수 없습니다. Team-6 저장소 안에서 노트북을 실행해 주세요.')

FIGURE_DIR = PROJECT_ROOT / 'output' / 'figures'
TABLE_DIR = PROJECT_ROOT / 'output' / 'tables'
REPORT_DIR = PROJECT_ROOT / 'output' / 'reports'
for directory in [FIGURE_DIR, TABLE_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def slugify(value):
    return str(value).lower().replace(' ', '_').replace('-', '_')


def save_current_figure(filename):
    path = FIGURE_DIR / filename
    plt.savefig(path, dpi=160, bbox_inches='tight')
    print(f'그래프 저장 완료: {path}')


## 1. 평가 함수 정의

세 모델을 같은 기준으로 비교하기 위해 Accuracy, F1, ROC-AUC, classification report, confusion matrix를 한 함수에서 계산합니다. Confusion matrix는 `output/figures/`에 저장합니다.


In [ ]:
# 세 가지 모델을 동일한 기준으로 공정하게 채점하기 위한 통합 성능 측정 함수입니다. 뒤에서 같은 거 3번 쓰는 것보다 함수로 만들면 편할 거 같아 이렇게 해보았습니다.

# 1차적인 성능 지표로서 정확도, 불균형 데이터에 중요한 F1 점수, 모델의 분류 성능인 ROC-AUC 값이 나오며,
# 이후 긍정과 부정별로 정밀도와 재현율을 볼 수 있도록 하고,
# 모델이 어떤 걸 맞추고 틀렸는지 볼 수 있는 혼동 행렬를 그려보았습니다.

def evaluate_model(y_true, y_pred, y_proba, model_name, city_name):
    print(f"\n[{city_name}] {model_name} 모델 성능 평가")
    print(f"정확도    : {accuracy_score(y_true, y_pred):.4f}")
    print(f"F1 점수   : {f1_score(y_true, y_pred):.4f}")
    print(f"ROC-AUC  : {roc_auc_score(y_true, y_proba):.4f}")

    print("\n[정밀도와 재현율]")
    print(classification_report(y_true, y_pred, target_names=['만족(0)', '불만족(1)']))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['예측: 만족', '예측: 불만족'],
                yticklabels=['실제: 만족', '실제: 불만족'])
    plt.title(f'{city_name} {model_name} 혼동 행렬')
    plt.ylabel('실제 라벨')
    plt.xlabel('예측 라벨')
    plt.tight_layout()
    save_current_figure(f"model_{slugify(city_name)}_{slugify(model_name)}_confusion_matrix.png")
    plt.show()

## 2. 데이터 로드 및 분할

로컬 `data/processed`와 `data/embeddings` 파일을 우선 사용하고, 로컬 파일이 없을 때만 GitHub raw URL을 fallback으로 사용합니다.


In [ ]:
# 로컬 CSV를 우선 사용하고, 파일이 없으면 GitHub 원격 파일을 예비 경로로 사용합니다.
# target_label은 불만족 리뷰를 1로 두기 위해 is_positive를 반전합니다.

data_sources = {
    "New Orleans": {
        "raw_path": PROJECT_ROOT / "data" / "processed" / "yelp_subset_new_orleans_15k_features.csv",
        "pca_path": PROJECT_ROOT / "data" / "embeddings" / "new_orleans_pca_32.csv",
        "raw_url": "https://raw.githubusercontent.com/2026-1st/Team-6/main/data/processed/yelp_subset_new_orleans_15k_features.csv",
        "pca_url": "https://raw.githubusercontent.com/2026-1st/Team-6/main/data/embeddings/new_orleans_pca_32.csv",
    },
    "Philly": {
        "raw_path": PROJECT_ROOT / "data" / "processed" / "yelp_subset_philly_15k_features.csv",
        "pca_path": PROJECT_ROOT / "data" / "embeddings" / "philly_pca_32.csv",
        "raw_url": "https://raw.githubusercontent.com/2026-1st/Team-6/main/data/processed/yelp_subset_philly_15k_features.csv",
        "pca_url": "https://raw.githubusercontent.com/2026-1st/Team-6/main/data/embeddings/philly_pca_32.csv",
    },
    "Tucson": {
        "raw_path": PROJECT_ROOT / "data" / "processed" / "yelp_subset_tucson_15k_features.csv",
        "pca_path": PROJECT_ROOT / "data" / "embeddings" / "tucson_pca_32.csv",
        "raw_url": "https://raw.githubusercontent.com/2026-1st/Team-6/main/data/processed/yelp_subset_tucson_15k_features.csv",
        "pca_url": "https://raw.githubusercontent.com/2026-1st/Team-6/main/data/embeddings/tucson_pca_32.csv",
    },
}

github_urls = data_sources

def read_csv_with_fallback(local_path, remote_url):
    if Path(local_path).exists():
        print(f'로컬 CSV 불러오기: {local_path}')
        return pd.read_csv(local_path)
    print(f'로컬 CSV가 없어 원격 CSV를 불러옵니다: {remote_url}')
    return pd.read_csv(remote_url)

target_col = 'is_positive'
city_datasets = {}
global_results = {city: {} for city in data_sources.keys()}

for city, sources in data_sources.items():
    df_raw = read_csv_with_fallback(sources["raw_path"], sources["raw_url"])
    df_pca = read_csv_with_fallback(sources["pca_path"], sources["pca_url"])

    df_pca['target_label'] = 1 - df_raw[target_col].values

    X = df_pca.drop(columns=['target_label'])
    y = df_pca['target_label']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    city_datasets[city] = (X_train, X_test, y_train, y_test)
    print(f"{city} 데이터 분할 완료 (학습: {X_train.shape[0]}건, 평가: {X_test.shape[0]}건)")


## 3. PyTorch MLP 구조 정의

32차원 PCA 임베딩을 입력으로 받는 MLP와 학습 루프를 정의합니다. 클래스 불균형은 `BCEWithLogitsLoss(pos_weight=...)`로 보정합니다.


In [ ]:
# PyTorch를 활용하여 딥러닝 모델의 아키텍처와 학습 함수를 정의하였습니다.
# 32차원으로 압축된 임베딩 데이터를 입력받아서 긍정/부정 확률을 출력하도록 3개의 층으로 구성하였고,
# 데이터의 클래스 불균형, 즉 불만족 리뷰가 너무 적거나 하는 등의 현상을 해결하기 위해서 불만족 리뷰를 놓쳤을 때 더 큰 가중치를 부여하도록 하는 비용 민감 학습인 BCEWithLogitsLoss를 적용하였습니다.
# 모델 학습이 끝나면 정의한 평가 함수를 호출하고, 추후 시각화를 위해 예측 확률을 저장하도록 하였습니다.

# 데이터셋 클래스 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class ReviewDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X.values, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32).unsqueeze(1)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

class MLP(nn.Module):
    def __init__(self, input_dim=32):
        super(MLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 1)
        )
    def forward(self, x): return self.net(x)

def train_and_evaluate_dl(city, X_train, X_test, y_train, y_test, epochs=30):
    train_loader = DataLoader(ReviewDataset(X_train, y_train), batch_size=64, shuffle=True)
    test_loader = DataLoader(ReviewDataset(X_test, y_test), batch_size=64, shuffle=False)
    model = MLP(input_dim=32).to(device)

    count_satisfied = sum(y_train == 0)
    count_dissatisfied = sum(y_train == 1)

    # BCEWithLogitsLoss의 pos_weight는 (부정 Class Count / 긍정 Class Count) 입니다
    pos_weight_val = torch.tensor([count_satisfied / count_dissatisfied], dtype=torch.float32).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_val)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    model.train()
    for epoch in range(epochs):
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(batch_X), batch_y)
            loss.backward()
            optimizer.step()

    model.eval()
    all_preds, all_probs = [], []
    with torch.no_grad():
        for batch_X, _ in test_loader:
            probs = torch.sigmoid(model(batch_X.to(device)))
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend((probs >= 0.5).float().cpu().numpy())

    preds_flat = np.array(all_preds).flatten()
    probs_flat = np.array(all_probs).flatten()

    evaluate_model(y_test, preds_flat, probs_flat, "PyTorch MLP", city)
    global_results[city]['dl_proba'] = probs_flat

## 4. 도시별 모델 학습

각 도시에서 Logistic Regression, CatBoost, PyTorch MLP를 학습하고 예측 확률을 `global_results`에 저장합니다.


In [ ]:
# 앞서 분할한 데이터 중 먼저 뉴올리언스 지역의 데이터를 바탕으로 세 가지 모델의 학습 및 평가를 진행하였습니다.
# 학습이 끝난 후, 각 모델의 예측 확률값과 CatBoost 모델 객체는 최종 비교 시각화 및 SHAP 분석을 위해 딕셔너리에 저장하였습니다.

city = "New Orleans"
X_train, X_test, y_train, y_test = city_datasets[city]

# 로지스틱 회귀
lr_model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)
lr_proba = lr_model.predict_proba(X_test)[:, 1]
evaluate_model(y_test, lr_preds, lr_proba, "로지스틱 회귀", city)
global_results[city]['lr_proba'] = lr_proba

# CatBoost
count_satisfied = sum(y_train == 0)
count_dissatisfied = sum(y_train == 1)

cb_model = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                              scale_pos_weight=(count_satisfied / count_dissatisfied),
                              eval_metric='AUC', random_seed=42, verbose=0)
cb_model.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
cb_preds = cb_model.predict(X_test)
cb_proba = cb_model.predict_proba(X_test)[:, 1]
evaluate_model(y_test, cb_preds, cb_proba, "CatBoost", city)
global_results[city]['cb_model'] = cb_model
global_results[city]['cb_proba'] = cb_proba

# PyTorch MLP
train_and_evaluate_dl(city, X_train, X_test, y_train, y_test, epochs=30)

In [ ]:
# 필라델피아 지역

city = "Philly"
X_train, X_test, y_train, y_test = city_datasets[city]

# 로지스틱 회귀
lr_model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)
lr_proba = lr_model.predict_proba(X_test)[:, 1]
evaluate_model(y_test, lr_preds, lr_proba, "로지스틱 회귀", city)
global_results[city]['lr_proba'] = lr_proba

# CatBoost
count_satisfied = sum(y_train == 0)
count_dissatisfied = sum(y_train == 1)

cb_model = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                              scale_pos_weight=(count_satisfied / count_dissatisfied),
                              eval_metric='AUC', random_seed=42, verbose=0)
cb_model.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
cb_preds = cb_model.predict(X_test)
cb_proba = cb_model.predict_proba(X_test)[:, 1]
evaluate_model(y_test, cb_preds, cb_proba, "CatBoost", city)
global_results[city]['cb_model'] = cb_model
global_results[city]['cb_proba'] = cb_proba

# PyTorch MLP
train_and_evaluate_dl(city, X_train, X_test, y_train, y_test, epochs=30)

In [ ]:
# 투손 지역

city = "Tucson"
X_train, X_test, y_train, y_test = city_datasets[city]

# 로지스틱 회귀
lr_model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)
lr_proba = lr_model.predict_proba(X_test)[:, 1]
evaluate_model(y_test, lr_preds, lr_proba, "로지스틱 회귀", city)
global_results[city]['lr_proba'] = lr_proba

# CatBoost
count_satisfied = sum(y_train == 0)
count_dissatisfied = sum(y_train == 1)

cb_model = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                              scale_pos_weight=(count_satisfied / count_dissatisfied),
                              eval_metric='AUC', random_seed=42, verbose=0)
cb_model.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
cb_preds = cb_model.predict(X_test)
cb_proba = cb_model.predict_proba(X_test)[:, 1]
evaluate_model(y_test, cb_preds, cb_proba, "CatBoost", city)
global_results[city]['cb_model'] = cb_model
global_results[city]['cb_proba'] = cb_proba

# PyTorch MLP
train_and_evaluate_dl(city, X_train, X_test, y_train, y_test, epochs=30)

## 5. 결과 시각화 및 해석

도시별 ROC 곡선, SHAP Summary/의존성 그래프, 종합 성능표, 오류 분석 결과를 생성하고 `output/`에 저장합니다.


In [ ]:
# 앞서 저장해둔 예측 결과와 모델 객체를 바탕으로 뉴올리언스 지역의 시각화 리포트를 생성하였습니다.
# 세 모델의 성능을 비교하는 ROC 곡선, 핵심 요인을 분석하는 SHAP 요약 그래프 및 의존성 그래프을 시각화하였습니다.
# 마지막으로 전체 데이터의 SHAP 값을 군집화(Clustering)하는 Heatmap Plot을 추가하여, 개별 리뷰가 아닌 '전체 고객의 불만 유형(거시 패턴)'이 어떻게 세분화되는지 거시적인 마케팅 인사이트를 도출하였습니다.

city = "New Orleans"
X_train, X_test, y_train, y_test = city_datasets[city]
lr_probs = global_results[city]['lr_proba']
cb_probs = global_results[city]['cb_proba']
dl_probs = global_results[city]['dl_proba']
cb_model = global_results[city]['cb_model']

# 다중 모델 ROC 곡선
plt.figure(figsize=(8, 6))
for name, probs, color in [("로지스틱 회귀", lr_probs, 'blue'),
                           ("CatBoost", cb_probs, 'green'),
                           ("PyTorch MLP", dl_probs, 'red')]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {auc(fpr, tpr):.4f})')

plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('오탐율 (False 긍정 Rate)')
plt.ylabel('재현율 (참 양성 Rate)')
plt.title('뉴올리언스 ROC 곡선')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
save_current_figure('new_orleans_roc_curve.png')
plt.show()

explainer = shap.TreeExplainer(cb_model)
shap_values = explainer.shap_values(X_test)

# SHAP 요약 그래프
plt.figure(figsize=(8, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title("뉴올리언스 피처 중요도")
plt.tight_layout()
save_current_figure('new_orleans_shap_summary.png')
plt.show()

# SHAP 의존성 그래프
top_feature_idx = np.argmax(np.abs(shap_values).mean(axis=0))
top_feature_name = X_test.columns[top_feature_idx]

plt.figure(figsize=(8, 5))
shap.dependence_plot(top_feature_idx, shap_values, X_test, interaction_index=None, show=False)
plt.axhline(0, color='red', linestyle='--', alpha=0.5, label='Threshold (SHAP=0)')
plt.title(f"뉴올리언스 핵심 피처 의존성: {top_feature_name}")
plt.legend()
plt.tight_layout()
save_current_figure('new_orleans_shap_dependence.png')
plt.show()

In [ ]:
# 필라델피아입니다.

city = "Philly"
X_train, X_test, y_train, y_test = city_datasets[city]
lr_probs = global_results[city]['lr_proba']
cb_probs = global_results[city]['cb_proba']
dl_probs = global_results[city]['dl_proba']
cb_model = global_results[city]['cb_model']

# 다중 모델 ROC 곡선
plt.figure(figsize=(8, 6))
for name, probs, color in [("로지스틱 회귀", lr_probs, 'blue'),
                           ("CatBoost", cb_probs, 'green'),
                           ("PyTorch MLP", dl_probs, 'red')]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {auc(fpr, tpr):.4f})')

plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('오탐율 (False 긍정 Rate)')
plt.ylabel('재현율 (참 양성 Rate)')
plt.title('필라델피아 ROC 곡선')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
save_current_figure('philly_roc_curve.png')
plt.show()

explainer = shap.TreeExplainer(cb_model)
shap_values = explainer.shap_values(X_test)

# SHAP 요약 그래프
plt.figure(figsize=(8, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title("필라델피아 피처 중요도")
plt.tight_layout()
save_current_figure('philly_shap_summary.png')
plt.show()

# SHAP 의존성 그래프
top_feature_idx = np.argmax(np.abs(shap_values).mean(axis=0))
top_feature_name = X_test.columns[top_feature_idx]

plt.figure(figsize=(8, 5))
shap.dependence_plot(top_feature_idx, shap_values, X_test, interaction_index=None, show=False)
plt.axhline(0, color='red', linestyle='--', alpha=0.5, label='Threshold (SHAP=0)')
plt.title(f"필라델피아 핵심 피처 의존성: {top_feature_name}")
plt.legend()
plt.tight_layout()
save_current_figure('philly_shap_dependence.png')
plt.show()


In [ ]:
# 투손입니다.

city = "Tucson"
X_train, X_test, y_train, y_test = city_datasets[city]
lr_probs = global_results[city]['lr_proba']
cb_probs = global_results[city]['cb_proba']
dl_probs = global_results[city]['dl_proba']
cb_model = global_results[city]['cb_model']

# 다중 모델 ROC 곡선
plt.figure(figsize=(8, 6))
for name, probs, color in [("로지스틱 회귀", lr_probs, 'blue'),
                           ("CatBoost", cb_probs, 'green'),
                           ("PyTorch MLP", dl_probs, 'red')]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {auc(fpr, tpr):.4f})')

plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('오탐율 (False 긍정 Rate)')
plt.ylabel('재현율 (참 양성 Rate)')
plt.title('투손 ROC 곡선')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
save_current_figure('tucson_roc_curve.png')
plt.show()

explainer = shap.TreeExplainer(cb_model)
shap_values = explainer.shap_values(X_test)

# SHAP 요약 그래프
plt.figure(figsize=(8, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title("투손 피처 중요도")
plt.tight_layout()
save_current_figure('tucson_shap_summary.png')
plt.show()

# SHAP 의존성 그래프
top_feature_idx = np.argmax(np.abs(shap_values).mean(axis=0))
top_feature_name = X_test.columns[top_feature_idx]

plt.figure(figsize=(8, 5))
shap.dependence_plot(top_feature_idx, shap_values, X_test, interaction_index=None, show=False)
plt.axhline(0, color='red', linestyle='--', alpha=0.5, label='Threshold (SHAP=0)')
plt.title(f"투손 핵심 피처 의존성: {top_feature_name}")
plt.legend()
plt.tight_layout()
save_current_figure('tucson_shap_dependence.png')
plt.show()

In [ ]:
# 3개 도시와 3가지 모델의 성능 지표(F1 점수, ROC-AUC)를 하나의 그래프로 통합하여 비교 분석하였습니다.
# 전역 딕셔너리에 저장된 예측 확률값을 활용하여 각 환경별 지표를 재산출하고, 이를 바 차트로 시각화하였습니다.

data_rows = []

for city in city_datasets.keys():
    _, _, _, y_test = city_datasets[city]

    lr_proba = global_results[city]['lr_proba']
    cb_proba = global_results[city]['cb_proba']
    dl_probs = global_results[city]['dl_proba']

    lr_pred = (lr_proba >= 0.5).astype(int)
    cb_pred = (cb_proba >= 0.5).astype(int)
    dl_pred = (dl_probs >= 0.5).astype(int)

    for model_name, preds, probas in [("로지스틱 회귀", lr_pred, lr_proba),
                                      ("CatBoost", cb_pred, cb_proba),
                                      ("PyTorch MLP", dl_pred, dl_probs)]:
        f1 = f1_score(y_test, preds)
        auc_val = roc_auc_score(y_test, probas)
        data_rows.append({"City": city, "Model": model_name, "F1 점수": f1, "ROC-AUC": auc_val})

df_metrics = pd.DataFrame(data_rows)
metrics_path = TABLE_DIR / 'model_metrics_summary.csv'
df_metrics.to_csv(metrics_path, index=False)
print(f'성능 요약표 저장 완료: {metrics_path}')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.barplot(x="City", y="F1 점수", hue="Model", data=df_metrics, ax=axes[0], palette="muted")
axes[0].set_title("도시별 모델 F1 점수 비교")
axes[0].set_ylim(0.7, 1.0)
axes[0].set_xlabel("도시")
axes[0].grid(axis='y', alpha=0.3)

sns.barplot(x="City", y="ROC-AUC", hue="Model", data=df_metrics, ax=axes[1], palette="muted")
axes[1].set_title("도시별 모델 ROC-AUC 비교")
axes[1].set_ylim(0.9, 1.05)
axes[1].set_xlabel("도시")
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
save_current_figure('model_metrics_comparison.png')
plt.show()

In [ ]:
# 모델이 예측에 실패한 원인을 규명하기 위해 도시별 오류 분석(오류 분석)을 수행하였습니다.
# 정답을 맞춘 불만족 리뷰(참 양성)와 만족 리뷰로 오해한 불만족 리뷰(거짓 음성)의 SHAP 값을 대조하여 모델이 분류에 혼선을 겪은 요인을 추적하였습니다.

for city in city_datasets.keys():
    X_train, X_test, y_train, y_test = city_datasets[city]
    cb_model = global_results[city]['cb_model']
    cb_preds = cb_model.predict(X_test)

    explainer = shap.TreeExplainer(cb_model)
    shap_values = explainer.shap_values(X_test)

    tp_idx = np.where((y_test == 1) & (cb_preds == 1))[0]
    fn_idx = np.where((y_test == 1) & (cb_preds == 0))[0]

    if len(fn_idx) > 0:
        tp_shap_mean = np.abs(shap_values[tp_idx]).mean(axis=0)
        fn_shap_mean = np.abs(shap_values[fn_idx]).mean(axis=0)

        df_error = pd.DataFrame({
            'Feature': X_test.columns,
            '정확한 불만 예측(TP)': tp_shap_mean,
            '오류 유발 불만 예측(FN)': fn_shap_mean
        }).sort_values(by='정확한 불만 예측(TP)', ascending=False).head(10)

        error_path = TABLE_DIR / f"error_analysis_{slugify(city)}.csv"
        df_error.to_csv(error_path, index=False)
        print(f'오류 분석표 저장 완료: {error_path}')

        df_error_melt = df_error.melt(id_vars='Feature', var_name='Group', value_name='Mean SHAP')

        plt.figure(figsize=(10, 5))
        sns.barplot(x='Mean SHAP', y='Feature', hue='Group', data=df_error_melt, palette='Set2')
        plt.title(f"[{city}] 예측 성공 및 실패 집단 간 SHAP 기여도 대조")
        plt.xlabel("SHAP 기여도 평균")
        plt.ylabel("핵심 피처")
        plt.tight_layout()
        save_current_figure(f"error_analysis_{slugify(city)}.png")
        plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer

# 특정 PCA Component의 고정값(높은 값/낮은 값)에 매핑되는 원본 텍스트를 추출하여 해당 차원의 언어학적/문맥적 의미를 해석하기 위한 역추적 분석 함수입니다.

def analyze_pca_meaning(city_name, pca_feature_name, top_n=5):
    urls = data_sources[city_name]
    df_raw = read_csv_with_fallback(urls["raw_path"], urls["raw_url"])
    df_pca = read_csv_with_fallback(urls["pca_path"], urls["pca_url"])

    text_col = 'text' if 'text' in df_raw.columns else df_raw.select_dtypes(include=[object]).columns[0]

    df_analysis = pd.DataFrame({
        'text': df_raw[text_col],
        'pca_value': df_pca[pca_feature_name]
    })

    top_reviews = df_analysis.sort_values(by='pca_value', ascending=False).head(top_n)
    bottom_reviews = df_analysis.sort_values(by='pca_value', ascending=True).head(top_n)

    print(f"[{city_name}] {pca_feature_name} 차원 해석")

    print(f"\n  [높은 값]")
    for row in top_reviews.itertuples():
        print(f"  ({row.pca_value:.4f}) {row.text[:120]}...")

    print(f"\n  [낮은 값]")
    for row in bottom_reviews.itertuples():
        print(f"  ({row.pca_value:.4f}) {row.text[:120]}...")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    vectorizer = CountVectorizer(stop_words='english', max_features=10)

    try:
        X_top = vectorizer.fit_transform(top_reviews['text'])
        top_words = pd.DataFrame(X_top.toarray(), columns=vectorizer.get_feature_names_out()).sum().sort_values(ascending=False)
        sns.barplot(x=top_words.values, y=top_words.index, ax=axes[0], palette='bone')
        axes[0].set_title(f"{pca_feature_name} (High)")
        axes[0].set_xlabel("빈도")
    except Exception:
        axes[0].text(0.5, 0.5, '데이터 부족', ha='center')

    try:
        X_bottom = vectorizer.fit_transform(bottom_reviews['text'])
        bottom_words = pd.DataFrame(X_bottom.toarray(), columns=vectorizer.get_feature_names_out()).sum().sort_values(ascending=False)
        sns.barplot(x=bottom_words.values, y=bottom_words.index, ax=axes[1], palette='bone')
        axes[1].set_title(f"{pca_feature_name} (Low)")
        axes[1].set_xlabel("빈도")
    except Exception:
        axes[1].text(0.5, 0.5, '데이터 부족', ha='center')

    plt.tight_layout()
    save_current_figure(f"pca_meaning_{slugify(city_name)}_{pca_feature_name}.png")
    plt.show()

In [ ]:
analyze_pca_meaning(city_name="New Orleans", pca_feature_name="roberta_pca_3", top_n=3)

In [ ]:
analyze_pca_meaning(city_name="Philly", pca_feature_name="roberta_pca_3", top_n=3)

In [ ]:
analyze_pca_meaning(city_name="Tucson", pca_feature_name="roberta_pca_3", top_n=3)